In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.


# SAM3D Objects + SAM2 (click prompt)

This notebook mirrors `demo_single_object.ipynb` but uses SAM2 click prompting to get a single-object mask.


## 1. Imports and Model Loading

In [ ]:
import os
import sys
import uuid
import imageio
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from IPython.display import Image as ImageDisplay

# Point to CUDA 12.8
os.environ["CUDA_HOME"] = "/usr/local/cuda-12.8"
os.environ["PATH"] = f"/usr/local/cuda-12.8/bin:{os.environ['PATH']}"
os.environ["FORCE_CUDA"] = "1"
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
print("CUDA_HOME", os.environ.get("CUDA_HOME"))

from torch.utils import cpp_extension
print("include_paths", cpp_extension.include_paths("cuda"))

# Use the same extensions dir you built to (adjust if you prefer a single location)
os.environ["TORCH_EXTENSIONS_DIR"] = "/home/mnc/mccv/sam-3d-objects/.torch_extensions"
os.makedirs(os.environ["TORCH_EXTENSIONS_DIR"], exist_ok=True)

# Notebook path setup
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.insert(0, os.getcwd())

from gsplat.cuda import _backend
print("gsplat CUDA loaded:", hasattr(_backend, "_C"))

from inference import (
    Inference,
    ready_gaussian_for_video_rendering,
    render_video,
    load_image,
    load_single_mask,
    display_image,
    make_scene,
    interactive_visualizer,
    debug_inference_and_save,
)


In [ ]:
PATH = os.getcwd()
TAG = "hf"
config_path = f"{PATH}/../checkpoints/{TAG}/pipeline.yaml"
inference = Inference(config_path, compile=False)


## 2. Select image

In [ ]:
# Pick an image to lift to 3D
IMAGE_PATH = "/home/mnc/Downloads/IMG_3792.jpg"
IMAGE_NAME = os.path.basename(os.path.dirname(IMAGE_PATH))

image = load_image(IMAGE_PATH)
display_image(image)


## 3. SAM2 click prompt to get a mask

In [ ]:
# SAM2 setup (choose backend: 'sam2_repo' or 'transformers')
BACKEND = os.environ.get("SAM2_BACKEND", "sam2_repo")
device = "cuda" if torch.cuda.is_available() else "cpu"

if BACKEND == "sam2_repo":
    # Requires the SAM2 repo cloned at SAM2_ROOT
    SAM2_ROOT = os.environ.get("SAM2_ROOT", "/home/mnc/mccv/sam2")
    SAM2_CHECKPOINT = os.environ.get("SAM2_CHECKPOINT", os.path.join(SAM2_ROOT, "checkpoints", "sam2.1_hiera_large.pt"))

    # Pick the matching config based on checkpoint name
    if os.path.basename(SAM2_CHECKPOINT).startswith("sam2.1_"):
        SAM2_CFG = os.environ.get("SAM2_CFG", "configs/sam2.1/sam2.1_hiera_l.yaml")
    else:
        SAM2_CFG = os.environ.get("SAM2_CFG", "configs/sam2/sam2_hiera_l.yaml")

    # Normalize SAM2_CFG if user provided a path or omitted configs/
    if os.path.isabs(SAM2_CFG):
        if "configs/" in SAM2_CFG:
            SAM2_CFG = SAM2_CFG.split("configs/")[-1]
            SAM2_CFG = "configs/" + SAM2_CFG
    if not SAM2_CFG.startswith("configs/"):
        SAM2_CFG = "configs/" + SAM2_CFG

    if not os.path.exists(SAM2_CHECKPOINT):
        raise FileNotFoundError(f"SAM2 checkpoint not found: {SAM2_CHECKPOINT}")

    sys.path.append(SAM2_ROOT)
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor

    sam2_model = build_sam2(SAM2_CFG, SAM2_CHECKPOINT, device=device)
    sam2_predictor = SAM2ImagePredictor(sam2_model)
    sam2_predictor.model.eval()
    print("SAM2 repo predictor ready on", device, "config", SAM2_CFG)

elif BACKEND == "transformers":
    # Transformers backend (requires Sam2* in transformers)
    MODEL_ID = os.environ.get("SAM2_MODEL_ID", "facebook/sam2-hiera-large")
    CACHE_DIR = os.environ.get("HF_HUB_CACHE", os.path.expanduser("~/.cache/huggingface"))
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

    try:
        from transformers import Sam2Processor, Sam2Model
    except Exception as e:
        raise ImportError("Sam2Processor/Sam2Model not available in this transformers build. Set SAM2_BACKEND='sam2_repo' instead.") from e

    model = Sam2Model.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR, token=HF_TOKEN, torch_dtype=(torch.float16 if device=="cuda" else None)).to(device).eval()
    processor = Sam2Processor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR, token=HF_TOKEN)
    print("SAM2 transformers ready on", device)

else:
    raise ValueError("BACKEND must be 'sam2_repo' or 'transformers'")


In [ ]:
# Prompting modes: 'points', 'box', or 'multi'
MODE = "points"
MULTIMASK_OUTPUT = True

# For MODE='points'
N_POS = 1
N_NEG = 0

# For MODE='multi' (provide your own points/labels per object)
# OBJECT_POINTS = [
#     [[500, 375], [1125, 625]],  # object 0
#     [[650, 750]],               # object 1
# ]
# OBJECT_LABELS = [
#     [1, 1],
#     [1],
# ]
OBJECT_POINTS = None
OBJECT_LABELS = None

# Mask selection
OBJECT_INDEX = 0
MASK_INDEX = None  # None -> pick best by score if available

raw_image = Image.fromarray(image).convert("RGB")

def get_points_cv2(img_rgb, n_pos, n_neg):
    try:
        import cv2
    except Exception as e:
        raise RuntimeError("cv2 not available. Install opencv-python or use ipympl for widget backend.") from e
    import numpy as _np

    # Ensure numpy uint8 RGB
    img_rgb = _np.asarray(img_rgb, dtype=_np.uint8)
    if img_rgb.ndim == 2:
        img_rgb = _np.repeat(img_rgb[..., None], 3, axis=2)
    if img_rgb.shape[-1] == 4:
        img_rgb = img_rgb[..., :3]
    img_rgb = _np.ascontiguousarray(img_rgb)
    # Avoid cv2.cvtColor issues in some notebook environments
    img_bgr = img_rgb[..., ::-1].copy()
    points = []
    labels = []

    def on_mouse(event, x, y, flags, param):
        nonlocal points, labels
        if event == cv2.EVENT_LBUTTONDOWN:
            points.append((x, y))
            labels.append(1)
        elif event == cv2.EVENT_RBUTTONDOWN:
            points.append((x, y))
            labels.append(0)

    win = 'Click: left=positive, right=negative (press q to finish)'
    cv2.namedWindow(win, cv2.WINDOW_NORMAL)
    cv2.setMouseCallback(win, on_mouse)

    while True:
        disp = _np.ascontiguousarray(img_bgr.copy())
        for (x, y), lab in zip(points, labels):
            color = (0, 255, 0) if lab == 1 else (0, 0, 255)
            cv2.circle(disp, (x, y), 4, color, -1)
        try:
            cv2.imshow(win, disp)
        except Exception as e:
            cv2.destroyWindow(win)
            raise RuntimeError("OpenCV window display failed; falling back to matplotlib.") from e
        key = cv2.waitKey(30) & 0xFF
        if key == ord('q'):
            break
        if n_pos is not None and labels.count(1) >= n_pos and n_neg is not None and labels.count(0) >= n_neg:
            # allow user to press q to finish early
            pass
    cv2.destroyWindow(win)
    return points, labels

def get_points_mpl(img_rgb, n_pos, n_neg):
    import matplotlib.pyplot as plt
    img_rgb = np.asarray(img_rgb, dtype=np.uint8)
    if img_rgb.ndim == 2:
        img_rgb = np.repeat(img_rgb[..., None], 3, axis=2)
    if img_rgb.shape[-1] == 4:
        img_rgb = img_rgb[..., :3]

    fig, ax = plt.subplots()
    ax.imshow(img_rgb)
    ax.set_title("Click: left=positive, right=negative; close window when done")
    points = []
    labels = []

    def onclick(event):
        if event.inaxes != ax:
            return
        if event.button == 1:
            points.append((int(event.xdata), int(event.ydata)))
            labels.append(1)
            ax.plot(event.xdata, event.ydata, 'go')
        elif event.button == 3:
            points.append((int(event.xdata), int(event.ydata)))
            labels.append(0)
            ax.plot(event.xdata, event.ydata, 'ro')
        fig.canvas.draw_idle()

    cid = fig.canvas.mpl_connect('button_press_event', onclick)
    plt.show()
    fig.canvas.mpl_disconnect(cid)
    return points, labels

# Prepare prompts
point_coords = None
point_labels = None
box = None

if MODE == "points":
    # Use OpenCV window to click points (left=positive, right=negative). Press 'q' to finish.
    try:
        pts, labels = get_points_cv2(raw_image, N_POS, N_NEG)
    except Exception:
        # Fallback for headless/Qt issues
        pts, labels = get_points_mpl(raw_image, N_POS, N_NEG)
    if len(pts) == 0:
        raise RuntimeError("No points selected.")
    point_coords = np.array(pts, dtype=np.float32)
    point_labels = np.array(labels, dtype=np.int32)

elif MODE == "box":
    # Click two corners (left clicks) and press 'q'
    pts, labels = get_points_cv2(raw_image, 2, 0)
    if len(pts) < 2:
        raise RuntimeError("Need 2 points for box.")
    (x0, y0), (x1, y1) = pts[:2]
    box = np.array([min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)], dtype=np.float32)

elif MODE == "multi":
    if OBJECT_POINTS is None or OBJECT_LABELS is None:
        raise RuntimeError("Set OBJECT_POINTS and OBJECT_LABELS for multi-object mode.")
    # Flatten all points/labels for repo predictor; also keep per-object lists for transformers
    point_coords = np.array([p for obj in OBJECT_POINTS for p in obj], dtype=np.float32)
    point_labels = np.array([l for obj in OBJECT_LABELS for l in obj], dtype=np.int32)

else:
    raise ValueError("MODE must be 'points', 'box', or 'multi'.")

if BACKEND == "sam2_repo":
    sam2_predictor.set_image(np.array(raw_image))
    masks, scores, logits = sam2_predictor.predict(
        point_coords=point_coords,
        point_labels=point_labels,
        box=box,
        multimask_output=MULTIMASK_OUTPUT,
    )
    # masks: (C,H,W), scores: (C,)
    if MASK_INDEX is None:
        mask_idx = int(np.argmax(scores)) if scores is not None else 0
    else:
        mask_idx = int(MASK_INDEX)
    mask = masks[mask_idx].astype(bool)

elif BACKEND == "transformers":
    # Build transformer prompt tensors
    input_points = None
    input_labels = None
    input_boxes = None
    if MODE in ("points", "multi"):
        if MODE == "multi":
            input_points = [OBJECT_POINTS]
            input_labels = [OBJECT_LABELS]
        else:
            input_points = [[point_coords.tolist()]]
            input_labels = [[point_labels.tolist()]]
    if MODE == "box":
        input_boxes = [[[float(box[0]), float(box[1]), float(box[2]), float(box[3])]]]

    inputs = processor(
        images=raw_image,
        input_points=input_points,
        input_labels=input_labels,
        input_boxes=input_boxes,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs, multimask_output=MULTIMASK_OUTPUT)

    masks = processor.post_process_masks(outputs.pred_masks.cpu(), inputs["original_sizes"])[0]
    if masks.ndim == 3:
        masks_by_obj = masks[None]
    else:
        masks_by_obj = masks

    scores = None
    if hasattr(outputs, "iou_scores"):
        scores = outputs.iou_scores.cpu().numpy()
    elif hasattr(outputs, "pred_iou"):
        scores = outputs.pred_iou.cpu().numpy()

    obj_idx = max(0, min(OBJECT_INDEX, masks_by_obj.shape[0] - 1))
    if scores is not None:
        scores_obj = scores if scores.ndim == 1 else scores[obj_idx]
        best_idx = int(scores_obj.argmax())
    else:
        best_idx = 0

    mask_idx = best_idx if MASK_INDEX is None else int(MASK_INDEX)
    mask = masks_by_obj[obj_idx, mask_idx].astype(bool)

else:
    raise ValueError("Invalid BACKEND")

# Save mask for SAM3D Objects pipeline (binary PNG)
mask_uint8 = (mask.astype(np.uint8) * 255)
mask_path = os.path.join(os.path.dirname(IMAGE_PATH), "0.png")
Image.fromarray(mask_uint8).save(mask_path)
print("Saved mask:", mask_path)

display_image(image, masks=[mask])


In [ ]:
# Show original image and saved mask
from PIL import Image
mask_img = np.array(Image.open(mask_path)) > 0
display_image(image, masks=[mask_img])


## 4. Generate Gaussian Splat (SAM3D Objects)

In [ ]:
# Run model
output = debug_inference_and_save(
    inference_fn=lambda img, msk, seed=42: inference(img, msk, seed=seed),
    image=image,
    mask=mask,
    out_dir=f"{PATH}/gaussians/single",
    image_name=IMAGE_NAME,
    seed=42,
)


## 5. Visualize Gaussian Splat

In [ ]:
# Render gaussian splat to GIF
os.makedirs(f"{PATH}/gaussians/single", exist_ok=True)

scene_gs = make_scene(output)
scene_gs = ready_gaussian_for_video_rendering(scene_gs)

video = render_video(
    scene_gs,
    r=1,
    fov=60,
    pitch_deg=15,
    yaw_start_deg=-45,
    resolution=512,
)["color"]

gif_path = os.path.join(PATH, "gaussians", "single", f"{IMAGE_NAME}.gif")
imageio.mimsave(gif_path, video, format="GIF", duration=1000 / 30, loop=0)

ImageDisplay(url=f"gaussians/single/{IMAGE_NAME}.gif?cache_invalidator={uuid.uuid4()}")


### Interactive Visualizer

In [ ]:
# Might take a while to load (black screen)
interactive_visualizer(f"{PATH}/gaussians/single/{IMAGE_NAME}.ply")
